In [1]:
#Imports 
import pandas as pd
import hashlib
import json

In [2]:
#Load and clean 
df = pd.read_excel('Network.xlsx')
df = df.apply(lambda x: x.str.strip().str.replace(r'\s+', ' ', regex=True) if x.dtype == "object" else x)
df = df.fillna('None')
df = df.replace('', 'None')
print(df.columns.tolist())
print(df.head())

['Person', 'Relationship', 'Met_Through', 'Met_Where', 'Industry', 'LinkedIn']
             Person Relationship Met_Through Met_Where      Industry  \
0                Me           Me        None      None            IT   
1    Myles Gascoyne       Family        None      None  Construction   
2  Belinda Gascoyne       Family        None      None        IP law   
3    Annie Gascoyne       Family        None      None  Consultancty   
4      Tom Gascoyne       Family        None      None  Architecture   

                                            LinkedIn  
0       www.linkedin.com/in/charlie-gascoyne20060619  
1  https://www.linkedin.com/in/myles-gascoyne-a83...  
2  https://www.linkedin.com/in/belinda-gascoyne-3...  
3         https://www.linkedin.com/in/anniegascoyne/  
4  https://www.linkedin.com/in/tom-gascoyne-22347...  


In [3]:
#Color coding and legends 
def industry_to_color(industry):
    if not industry or industry in ('None', 'nan', 'Unknown'):
        return '#888888'
    h = hashlib.sha256(industry.strip().lower().encode()).digest()
    r, g, b = h[0], h[1], h[2]
    return f'#{r:02x}{g:02x}{b:02x}'

# Build legends
Industry_legend = {}
for industry in df['Industry'].dropna().unique():
    if industry not in ('None', 'nan', ''):
        Industry_legend[industry] = industry_to_color(industry)

legend_items = ""
for industry, color in Industry_legend.items():
    legend_items += f'<div style="display:flex;align-items:center;margin-bottom:6px;"><span style="background:{color};width:12px;height:12px;border-radius:50%;display:inline-block;margin-right:8px;flex-shrink:0;"></span><span>{industry}</span></div>'

Relationship_legend = sorted(list(df['Relationship'].dropna().unique()))
Met_Through_legend  = sorted([x for x in df['Met_Through'].dropna().unique() if x != 'None'])
Met_Where_legend    = sorted([x for x in df['Met_Where'].dropna().unique() if x != 'None'])

print(f"Industries: {len(Industry_legend)}, Relationships: {len(Relationship_legend)}")

Industries: 22, Relationships: 7


In [4]:
#Build nodes and edges 
nodes_list = []
edges_list = []
added_nodes = set()

# Central "Me" node
nodes_list.append({
    'id': 'Me',
    'label': 'Me',
    'color': {'background': '#ffffff', 'border': '#cccccc'},
    'size': 25,
    'font': {'size': 14, 'color': '#ffffff', 'bold': True},
    'title': 'You',
    'linkedin': '',
    'industry': 'IT',
    'relationship': 'Me',
    'met_through': 'None',
    'met_where': 'None'
})
added_nodes.add('Me')

for _, row in df.iterrows():
    name        = str(row['Person']).strip()
    industry    = str(row['Industry']).strip()
    relationship= str(row['Relationship']).strip()
    met_through = str(row['Met_Through']).strip()
    met_where   = str(row['Met_Where']).strip()
    linkedin    = str(row['LinkedIn']).strip() if 'LinkedIn' in row else 'None'

    if not name or name in ('nan', 'None', 'Me'):
        continue

    color = industry_to_color(industry)
    is_direct = met_through in ('None', 'nan', '')

    # Tooltip HTML
    tooltip = f"<div style='font-family:Arial;padding:8px;'>"
    tooltip += f"<b style='font-size:14px;'>{name}</b><br>"
    if industry not in ('None', 'nan'):
        tooltip += f"<span style='color:{color};'>●</span> {industry}<br>"
    if relationship not in ('None', 'nan'):
        tooltip += f"🤝 {relationship}<br>"
    if met_where not in ('None', 'nan'):
        tooltip += f"📍 {met_where}<br>"
    if not is_direct:
        tooltip += f"👋 Introduced by {met_through}<br>"
    if linkedin not in ('None', 'nan', ''):
        tooltip += f"<a href='{linkedin}' style='color:#0077b5;'>LinkedIn →</a>"
    tooltip += "</div>"

    if name not in added_nodes:
        nodes_list.append({
            'id': name,
            'label': name,
            'color': {'background': color, 'border': color},
            'size': 16,
            'font': {'size': 11, 'color': '#ffffff'},
            'title': tooltip,
            'linkedin': linkedin if linkedin not in ('None', 'nan', '') else '',
            'industry': industry,
            'relationship': relationship,
            'met_through': met_through,
            'met_where': met_where
        })
        added_nodes.add(name)

    # Edge colour: gold if direct, white dashed if introduced
    # Edge colour: gold if direct, white dashed if introduced
    if is_direct:
        edges_list.append({
            'from': 'Me',
            'to': name,
            'color': {'color': '#FFD700', 'opacity': 0.9},
            'width': 1.5,
            'dashes': False
        })
    else:
        # Introduced - only connect to introducer, not to Me
        if met_through in added_nodes:
            edges_list.append({
                'from': met_through,
                'to': name,
                'color': {'color': '#ffffff', 'opacity': 0.6},
                'width': 1,
                'dashes': True
            })

nodes_json = json.dumps(nodes_list)
edges_json = json.dumps(edges_list)

rel_options      = "".join(f'<option value="{r}">{r}</option>' for r in sorted(set(Relationship_legend)))
through_options  = "".join(f'<option value="{m}">{m}</option>' for m in sorted(Met_Through_legend))
where_options    = "".join(f'<option value="{m}">{m}</option>' for m in sorted(Met_Where_legend))
industry_options = "".join(f'<option value="{i}">{i}</option>' for i in sorted(Industry_legend.keys()))

print(f"✅ {len(nodes_list)} nodes, {len(edges_list)} edges")

# ── HTML ──────────────────────────────────────────────────────────────────────
html = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>My Professional Network</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/vis-network/9.1.9/dist/vis-network.min.js"></script>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ background: #1a1a2e; font-family: Arial, sans-serif; display: flex; flex-direction: column; height: 100vh; overflow: hidden; }}
  #header {{ padding: 12px 20px; background: #1a1a2e; border-bottom: 1px solid #2e3248; }}
  #header h1 {{ color: white; font-size: 22px; margin-bottom: 10px; }}
  #filters {{ display: flex; gap: 10px; flex-wrap: wrap; align-items: center; }}
  #filters input, #filters select {{
    padding: 6px 10px; background: #16213e; color: white;
    border: 1px solid #2e3248; border-radius: 6px; font-size: 13px; outline: none;
  }}
  #filters input:focus, #filters select:focus {{ border-color: #FFD700; }}
  #legend {{
    position: fixed; top: 90px; right: 16px; background: #16213e;
    padding: 12px 16px; border-radius: 8px; color: white; font-size: 13px;
    z-index: 999; max-height: calc(100vh - 110px); overflow-y: auto;
    border: 1px solid #2e3248;
  }}
  #legend h3 {{ font-size: 13px; margin-bottom: 8px; color: #aaa; }}
  #network {{ width: 100%; height: calc(100vh - 80px); }}
</style>
</head>
<body>

<div id="header">
  <h1>My Professional Network</h1>
  <div id="filters">
    <input type="text" id="searchInput" placeholder="Search name..." style="width:160px;">
    <select id="industryFilter">
      <option value="">All Industries</option>
      {industry_options}
    </select>
    <select id="relationshipFilter">
      <option value="">All Relationships</option>
      {rel_options}
    </select>
    <select id="metThroughFilter">
      <option value="">Met through anyone</option>
      {through_options}
    </select>
    <select id="metWhereFilter">
      <option value="">Met anywhere</option>
      {where_options}
    </select>
  </div>
</div>

<div id="legend">
  <h3>Industries</h3>
  {legend_items}
</div>

<div id="network" style="width:100%; height:calc(100vh - 80px);"></div>

<script>
const nodes_data = {nodes_json};
const edges_data = {edges_json};

const nodesDS = new vis.DataSet(nodes_data);
const edgesDS = new vis.DataSet(edges_data);

const container = document.getElementById('network');

const options = {{
  physics: {{
    solver: 'barnesHut',
    barnesHut: {{
      gravitationalConstant: -8000,
      springLength: 250,
      centralGravity: 0.05,
      springConstant: 0.02,
      damping: 0.15
    }},
    stabilization: {{ iterations: 500, fit: true }}
  }},
  nodes: {{
    shape: 'ellipse',
    borderWidth: 1.5,
    shadow: {{ enabled: true, color: 'rgba(0,0,0,0.4)', size: 6 }},
    font: {{ color: '#ffffff', size: 11, face: 'Arial' }}
  }},
  edges: {{
    smooth: {{ type: 'cubicBezier', forceDirection: 'none', roundness: 0.4 }},
    arrows: {{ to: {{ enabled: false }} }}
  }},
  interaction: {{
    hover: true,
    tooltipDelay: 80,
    hideEdgesOnDrag: true
  }}
}};

const network = new vis.Network(container, {{ nodes: nodesDS, edges: edgesDS }}, options);

network.on('stabilized', function() {{
    network.setOptions({{ physics: {{ enabled: false }} }});
}});

// Click node → open LinkedIn
network.on('click', function(params) {{
  if (params.nodes.length > 0) {{
    const nodeId = params.nodes[0];
    const node = nodes_data.find(n => n.id === nodeId);
    if (node && node.linkedin) {{
      window.open(node.linkedin, '_blank');
    }}
  }}
}});

// Pointer cursor on hover
network.on('hoverNode', function() {{
  container.style.cursor = 'pointer';
}});
network.on('blurNode', function() {{
  container.style.cursor = 'default';
}});

// ── Filters ──────────────────────────────────────────────────────────────────
function applyFilters() {{
  const search       = document.getElementById('searchInput').value.toLowerCase().trim();
  const industry     = document.getElementById('industryFilter').value;
  const relationship = document.getElementById('relationshipFilter').value;
  const metThrough   = document.getElementById('metThroughFilter').value;
  const metWhere     = document.getElementById('metWhereFilter').value;

  const noFilters = !search && !industry && !relationship && !metThrough && !metWhere;

  const visibleIds    = new Set();
  const connectorIds  = new Set();
  visibleIds.add('Me');

  if (noFilters) {{
    nodes_data.forEach(n => visibleIds.add(n.id));
  }} else {{
    nodes_data.forEach(n => {{
      if (n.id === 'Me') return;
      const matchSearch       = !search       || n.label.toLowerCase().includes(search);
      const matchIndustry     = !industry     || n.industry === industry;
      const matchRelationship = !relationship || n.relationship === relationship;
      const matchMetThrough   = !metThrough   || n.met_through === metThrough;
      const matchMetWhere     = !metWhere     || n.met_where === metWhere;
      if (matchSearch && matchIndustry && matchRelationship && matchMetThrough && matchMetWhere) {{
        visibleIds.add(n.id);
      }}
    }});

    edges_data.forEach(e => {{
      if (visibleIds.has(e.to) && !visibleIds.has(e.from) && e.from !== 'Me') {{
        connectorIds.add(e.from);
      }}
    }});
  }}

  nodesDS.update(nodes_data.map(n => {{
    const isConnector = connectorIds.has(n.id);
    const isVisible   = visibleIds.has(n.id);
    return {{
      id: n.id,
      hidden: !isVisible && !isConnector,
      color: isConnector
        ? {{ background: '#444444', border: '#666666' }}
        : {{ background: n.color.background, border: n.color.border }},
      opacity: isConnector ? 0.4 : 1.0
    }};
  }}));

  // Remove old synthetic edges first
  const oldSyn = edgesDS.get().filter(e => typeof e.id === 'string' && e.id.startsWith('syn_'));
  if (oldSyn.length) edgesDS.remove(oldSyn.map(e => e.id));

  // Add synthetic dashed Me→connector edges so chain is visible
  connectorIds.forEach(cid => {{
    const alreadyLinked = edges_data.some(e =>
      (e.from === 'Me' && e.to === cid) || (e.from === cid && e.to === 'Me')
    );
    if (!alreadyLinked) {{
      edgesDS.add({{
        id: 'syn_' + cid,
        from: 'Me',
        to: cid,
        color: {{ color: '#ffffff', opacity: 0.3 }},
        width: 1,
        dashes: true
      }});
    }}
  }});

  edgesDS.update(edges_data.map((e, i) => {{
    const fromOk = visibleIds.has(e.from) || connectorIds.has(e.from);
    const toOk   = visibleIds.has(e.to)   || connectorIds.has(e.to);
    return {{ id: i, hidden: !fromOk || !toOk }};
  }}));
}}

document.getElementById('searchInput').addEventListener('input', applyFilters);
document.getElementById('industryFilter').addEventListener('change', applyFilters);
document.getElementById('relationshipFilter').addEventListener('change', applyFilters);
document.getElementById('metThroughFilter').addEventListener('change', applyFilters);
document.getElementById('metWhereFilter').addEventListener('change', applyFilters);
</script>
</body>
</html>"""

with open('final_network.html', 'w') as f:
    f.write(html)

print("✅ final_network.html generated successfully")

✅ 43 nodes, 42 edges
✅ final_network.html generated successfully
